### Advanced feature engineering


In [15]:
import pandas as pd
import numpy as np

In [16]:
men_df = pd.read_csv("../data/processed/m_tournament_training_dataset.csv")
women_df = pd.read_csv("../data/processed/w_tournament_training_dataset.csv")

In [17]:
print("Initial shape:", men_df.shape)
print("\nColumns:")
print(men_df.columns.tolist())

Initial shape: (2898, 20)

Columns:
['Season', 'Team1ID', 'Team2ID', 'Target', 'WinPctDiff', 'SeedNumDiff', 'NetRatingDiff', 'OffEffDiff', 'DefEffDiff', 'MarginDiff', 'ReboundPctDiff', 'TurnoverPctDiff', 'FGPctDiff', 'ThreePctDiff', 'FTPctDiff', 'RankingDiff', 'PossessionsDiff', 'TurnoverMarginDiff', 'ReboundMarginDiff', 'AssistTurnoverRatioDiff']


Helper function to avoid divide-by-zero issues

In [18]:
def safe_divide(a, b):
    return np.where(np.abs(b) < 1e-8, 0, a / b)

### Let's create now advanced engineered features

In [19]:
def add_advanced_features(df):
    df = df.copy()

    assist_turnover_col = (
        "AssistTurnoverRatioDiff"
        if "AssistTurnoverRatioDiff" in df.columns
        else "AssistTurnoverRtoDiff"
    )

    # Strength interaction features
    df["Seed_Rank_Interaction"] = df["SeedNumDiff"] * df["RankingDiff"]
    df["WinPct_NetRating_Interaction"] = df["WinPctDiff"] * df["NetRatingDiff"]
    df["Margin_Ranking_Interaction"] = df["MarginDiff"] * df["RankingDiff"]

    # Efficiency-based interactions
    df["OffDefGap"] = df["OffEffDiff"] - df["DefEffDiff"]
    df["TotalEfficiencyGap"] = df["OffEffDiff"] + df["DefEffDiff"]
    df["NetRating_Margin_Interaction"] = df["NetRatingDiff"] * df["MarginDiff"]
    df["OffEff_DefEff_Product"] = df["OffEffDiff"] * df["DefEffDiff"]

    # Shooting profile interactions
    df["ShootingEfficiencyScore"] = (
            df["FGPctDiff"] +
            df["ThreePctDiff"] +
            df["FTPctDiff"]
    )

    df["WeightedShootingScore"] = (
            0.5 * df["FGPctDiff"] +
            0.3 * df["ThreePctDiff"] +
            0.2 * df["FTPctDiff"]
    )

    df["FG_Three_Interaction"] = df["FGPctDiff"] * df["ThreePctDiff"]
    df["FG_FT_Interaction"] = df["FGPctDiff"] * df["FTPctDiff"]
    df["Three_FT_Interaction"] = df["ThreePctDiff"] * df["FTPctDiff"]

    # Ball control
    df["ControlScore"] = (
            df["TurnoverPctDiff"] +
            df["TurnoverMarginDiff"] +
            df[assist_turnover_col]
    )

    df["ReboundControlScore"] = (
            df["ReboundPctDiff"] +
            df["ReboundMarginDiff"]
    )

    df["PossessionControlInteraction"] = (
            df["PossessionsDiff"] * df["TurnoverPctDiff"]
    )

    df["ReboundTurnoverCombo"] = (
            df["ReboundPctDiff"] - df["TurnoverPctDiff"]
    )

    # Dominance features
    df["DominanceScore"] = df["MarginDiff"] * df["WinPctDiff"]
    df["SeedAdjustedDominance"] = safe_divide(df["MarginDiff"], df["SeedNumDiff"])
    df["RankingAdjustedNetRating"] = safe_divide(df["NetRatingDiff"], df["RankingDiff"])

    # Nonlinear features
    df["SeedNumDiff_Squared"] = df["SeedNumDiff"] ** 2
    df["NetRatingDiff_Squared"] = df["NetRatingDiff"] ** 2
    df["MarginDiff_Squared"] = df["MarginDiff"] ** 2
    df["RankingDiff_Squared"] = df["RankingDiff"] ** 2

    df["AbsSeedNumDiff"] = df["SeedNumDiff"].abs()
    df["AbsNetRatingDiff"] = df["NetRatingDiff"].abs()
    df["AbsMarginDiff"] = df["MarginDiff"].abs()
    df["AbsRankingDiff"] = df["RankingDiff"].abs()

    # Threshold / indicator features
    df["SeedAdvantageFlag"] = (df["SeedNumDiff"] < 0).astype(int)
    df["NetRatingAdvantageFlag"] = (df["NetRatingDiff"] > 0).astype(int)
    df["OffEffAdvantageFlag"] = (df["OffEffDiff"] > 0).astype(int)
    df["DefEffAdvantageFlag"] = (df["DefEffDiff"] < 0).astype(int)
    df["MarginAdvantageFlag"] = (df["MarginDiff"] > 0).astype(int)
    df["RankingAdvantageFlag"] = (df["RankingDiff"] < 0).astype(int)

    # Combined advantage count
    df["AdvantageCount"] = (
            df["SeedAdvantageFlag"] +
            df["NetRatingAdvantageFlag"] +
            df["OffEffAdvantageFlag"] +
            df["DefEffAdvantageFlag"] +
            df["MarginAdvantageFlag"] +
            df["RankingAdvantageFlag"]
    )

    df.replace([np.inf, -np.inf], 0, inplace=True)

    return df

In [20]:
men_df_advanced = add_advanced_features(men_df)
women_df_advanced = add_advanced_features(women_df)

### Save output


In [21]:
men_df_advanced.to_csv("../data/processed/m_tournament_training_dataset_advanced.csv", index=False)
women_df_advanced.to_csv("../data/processed/w_tournament_training_dataset_advanced.csv", index=False)